# TCIA Brain-Mets-Lung WSI Preprocessing
Extracts 256×256 tiles at effective 20× magnification from 99 NSCLC SVS files.
- Otsu thresholding for tissue segmentation
- Laplacian blur filter to drop blurry tiles
- SPCN stain normalization (Vahadane) using Zhou et al. reference tile
- Saves one .h5 file per slide with tiles + coordinates
- Checkpoint system: skips already-processed slides on re-run

**Run order: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8**
(Cell 4 is reference tile check only — takes 2 seconds)

In [ ]:
# ── CELL 1: CONFIGURATION ─────────────────────────────────────────────────────
SVS_DIR    = "/home/mth22/mtech_proj/brain-histo"        # folder with .svs files
OUTPUT_DIR = "/home/mth22/mtech_proj/brain-histo-tiles"  # where .h5 files are saved
REF_TILE   = "/home/mth22/mtech_proj/train/BM/6.png"     # Zhou reference tile for SPCN

TILE_SIZE           = 256    # pixels at 20x — matches Zhou et al.
TISSUE_THRESHOLD    = 0.80   # min tissue fraction per tile
MIN_LAP_VAR         = 100.0  # laplacian variance — below = blurry, skip
MAX_TILES_PER_SLIDE = None   # None = keep all valid tiles; set e.g. 5000 to cap
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── CELL 2: IMPORTS + LIST SVS FILES ──────────────────────────────────────────
import os, gc, time, warnings, json
import numpy as np
import h5py
import openslide
import cv2
import staintools
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

svs_files = sorted([f for f in os.listdir(SVS_DIR) if f.endswith('.svs')])
print(f"Found {len(svs_files)} SVS files")
for f in svs_files[:5]:
    size_mb = os.path.getsize(os.path.join(SVS_DIR, f)) / 1e6
    print(f"  {f}  ({size_mb:.0f} MB)")

In [ ]:
# ── CELL 3: INSPECT ONE SLIDE — CONFIRM MAGNIFICATION ─────────────────────────
sample_path = os.path.join(SVS_DIR, svs_files[0])
slide = openslide.OpenSlide(sample_path)

print("Level count      :", slide.level_count)
print("Level dimensions :", slide.level_dimensions)
print("Level downsamples:", [round(d, 2) for d in slide.level_downsamples])
print("Relevant properties:")
for k, v in slide.properties.items():
    if any(x in k.lower() for x in ['magnif', 'mpp', 'objective']):
        print(f"  {k}: {v}")
slide.close()

# TCIA slides are native 20x — level 0 IS 20x, READ_SIZE = TILE_SIZE = 256.
# If objective-power = 40, change READ_SIZE to 512 below.
LEVEL     = 0
READ_SIZE = TILE_SIZE  # 256 for native 20x; 512 for native 40x
print(f"\nUsing level={LEVEL}, READ_SIZE={READ_SIZE}")

In [ ]:
# ── CELL 4: CHECK REFERENCE TILE ──────────────────────────────────────────────
# Quick visual check — confirm it looks like a proper H&E tile.
ref_img_pil = Image.open(REF_TILE).convert('RGB')
print(f"Reference tile size : {ref_img_pil.size}")
print(f"Reference tile mode : {ref_img_pil.mode}")

plt.figure(figsize=(4, 4))
plt.imshow(ref_img_pil)
plt.title(f"Reference tile: {Path(REF_TILE).name} (Zhou)", fontsize=9)
plt.axis('off')
plt.show()

In [ ]:
# ── CELL 5: FIT SPCN NORMALIZER ON ZHOU REFERENCE TILE ───────────────────────
ref_img    = np.array(ref_img_pil)
normalizer = staintools.StainNormalizer(method='vahadane')
normalizer.fit(ref_img)
print("Stain normalizer fitted on Zhou reference tile.")
print("All TCIA tiles will be normalized to this color space.")

In [ ]:
# ── CELL 6: HELPER FUNCTIONS ──────────────────────────────────────────────────

def get_tissue_mask(slide):
    """
    Otsu tissue mask at lowest-res level.
    Returns (mask, thumb_w, thumb_h).
    """
    thumb_level = slide.level_count - 1
    tw, th = slide.level_dimensions[thumb_level]
    thumb  = np.array(slide.get_thumbnail((tw, th)).convert('RGB'))
    gray   = cv2.cvtColor(thumb, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255,
                            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((3, 3), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)
    return mask, tw, th


def tissue_fraction(tile_np, white_thresh=220):
    """Fraction of non-white pixels."""
    gray = cv2.cvtColor(tile_np, cv2.COLOR_RGB2GRAY)
    return float(np.mean(gray < white_thresh))


def is_blurry(tile_np):
    """True if Laplacian variance below threshold."""
    gray = cv2.cvtColor(tile_np, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var() < MIN_LAP_VAR


def safe_normalize(tile_np):
    """SPCN normalize; return original tile if normalization fails."""
    try:
        return normalizer.transform(tile_np)
    except Exception:
        return tile_np


print("Helper functions defined.")

In [ ]:
# ── CELL 7: MAIN PROCESSING LOOP ──────────────────────────────────────────────
# - Processes one slide at a time (memory safe for 16GB RAM)
# - Checkpoint: skips slides whose .h5 already exists
# - Saves .h5 immediately after each slide
# - Frees memory explicitly after every slide
# Expected time: ~3-8 min per slide, ~5-13 hours total for 99 slides

log = []

for idx, fname in enumerate(svs_files):
    slide_id = Path(fname).stem
    out_h5   = os.path.join(OUTPUT_DIR, f"{slide_id}.h5")

    # ── checkpoint ───────────────────────────────────────────────────────────
    if os.path.exists(out_h5):
        print(f"[{idx+1:02d}/{len(svs_files)}] SKIP (exists): {fname}")
        log.append({'slide': fname, 'status': 'skipped'})
        continue

    t0 = time.time()
    print(f"[{idx+1:02d}/{len(svs_files)}] {fname}", end=' ')

    tiles_list  = []
    coords_list = []

    try:
        slide  = openslide.OpenSlide(os.path.join(SVS_DIR, fname))
        w0, h0 = slide.level_dimensions[LEVEL]
        ds     = int(round(slide.level_downsamples[LEVEL]))

        mask, tw, th = get_tissue_mask(slide)

        n_low, n_blur = 0, 0

        for row in range(0, h0 - READ_SIZE + 1, READ_SIZE):
            for col in range(0, w0 - READ_SIZE + 1, READ_SIZE):

                # map tile centre to thumbnail mask coords
                mx = min(int((col + READ_SIZE // 2) * tw / w0), tw - 1)
                my = min(int((row + READ_SIZE // 2) * th / h0), th - 1)
                if mask[my, mx] == 0:
                    continue

                # level-0 origin for read_region
                x0_l0 = col * ds
                y0_l0 = row * ds

                region  = slide.read_region((x0_l0, y0_l0), LEVEL,
                                            (READ_SIZE, READ_SIZE))
                tile_np = np.array(region.convert('RGB'))

                # downsample if native 40x (READ_SIZE=512 → 256)
                if READ_SIZE != TILE_SIZE:
                    tile_np = cv2.resize(tile_np, (TILE_SIZE, TILE_SIZE),
                                         interpolation=cv2.INTER_AREA)

                # quality filters
                if tissue_fraction(tile_np) < TISSUE_THRESHOLD:
                    n_low += 1
                    continue
                if is_blurry(tile_np):
                    n_blur += 1
                    continue

                tile_norm = safe_normalize(tile_np)
                tiles_list.append(tile_norm.astype(np.uint8))
                coords_list.append([x0_l0, y0_l0])

                if MAX_TILES_PER_SLIDE and len(tiles_list) >= MAX_TILES_PER_SLIDE:
                    break
            if MAX_TILES_PER_SLIDE and len(tiles_list) >= MAX_TILES_PER_SLIDE:
                break

        slide.close()

        if len(tiles_list) == 0:
            print("-> 0 tiles passed filters, skipping")
            log.append({'slide': fname, 'status': 'all_filtered', 'n_tiles': 0})
            continue

        # ── write .h5 ────────────────────────────────────────────────────────
        tiles_arr  = np.stack(tiles_list,  axis=0)          # (N, 256, 256, 3)
        coords_arr = np.array(coords_list, dtype=np.int32)  # (N, 2)

        with h5py.File(out_h5, 'w') as f:
            f.create_dataset('tiles',  data=tiles_arr,
                             chunks=(1, TILE_SIZE, TILE_SIZE, 3),
                             compression='gzip', compression_opts=4)
            f.create_dataset('coords', data=coords_arr)
            f.attrs['slide_id']  = slide_id
            f.attrs['n_tiles']   = len(tiles_list)
            f.attrs['tile_size'] = TILE_SIZE
            f.attrs['level']     = LEVEL
            f.attrs['ref_tile']  = REF_TILE

        elapsed = time.time() - t0
        print(f"-> {len(tiles_list)} tiles "
              f"[{n_low} low-tissue, {n_blur} blurry filtered] "
              f"{elapsed:.0f}s")
        log.append({'slide': fname, 'status': 'ok',
                    'n_tiles': len(tiles_list),
                    'n_low': n_low, 'n_blur': n_blur,
                    'time_s': round(elapsed)})

    except Exception as e:
        print(f"-> ERROR: {e}")
        log.append({'slide': fname, 'status': 'error', 'error': str(e)})

    finally:
        # always free memory before next slide
        try: slide.close()
        except: pass
        tiles_list  = []
        coords_list = []
        gc.collect()

print("\nAll slides done.")

In [ ]:
# ── CELL 8: SUMMARY + SAVE LOG ────────────────────────────────────────────────
ok      = [l for l in log if l['status'] == 'ok']
skipped = [l for l in log if l['status'] == 'skipped']
errors  = [l for l in log if l['status'] == 'error']

print(f"Processed  : {len(ok)}")
print(f"Skipped    : {len(skipped)} (already existed)")
print(f"Errors     : {len(errors)}")
print(f"Total tiles: {sum(l['n_tiles'] for l in ok):,}")

if ok:
    counts = [l['n_tiles'] for l in ok]
    print(f"Tiles/slide: min={min(counts)}, "
          f"max={max(counts)}, "
          f"median={int(np.median(counts))}")

if errors:
    print("\nErrors:")
    for e in errors:
        print(f"  {e['slide']}: {e.get('error', '')}")

log_path = os.path.join(OUTPUT_DIR, 'preprocessing_log.json')
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2)
print(f"\nLog saved to: {log_path}")

In [ ]:
# ── CELL 9: VISUAL SANITY CHECK ───────────────────────────────────────────────
# Shows 16 random tiles from the first processed slide.
# Confirm tiles look like H&E tissue, not background or artifacts.

h5_files  = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.h5')])
check_h5  = os.path.join(OUTPUT_DIR, h5_files[0])

with h5py.File(check_h5, 'r') as f:
    tiles  = f['tiles'][:]
    coords = f['coords'][:]
    print(f"Slide  : {f.attrs['slide_id']}")
    print(f"Tiles  : {f.attrs['n_tiles']}")
    print(f"Shape  : {tiles.shape}")
    print(f"Ref    : {f.attrs['ref_tile']}")

idxs = np.random.choice(len(tiles), min(16, len(tiles)), replace=False)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    if i < len(idxs):
        ax.imshow(tiles[idxs[i]])
        ax.set_title(f"{coords[idxs[i], 0]},{coords[idxs[i], 1]}",
                     fontsize=7)
    ax.axis('off')
plt.suptitle(f"Sample tiles — {Path(check_h5).stem}", y=1.01)
plt.tight_layout()
plt.show()

del tiles
gc.collect()